# Emotion Recognition: Dynamic Images vs Video vs Static Images## Comprehensive Implementation with XAI Analysis### Dataset Structure- **Train/Test folders** with 7 emotion classes- **Classes**: neutral, surprise, sad, happy, fear, disgust, angry  - **Format**: MP4 videos (3-16 seconds)### Implementation Features1. Face detection and cropping (MTCNN/Haar Cascade)2. Three approaches: Dynamic Images, Video-LSTM, Static Images3. Multiple architectures: ResNet, VGG, MobileNet4. Comprehensive metrics: Top-1 Accuracy, Macro-F1, Per-class ROC-AUC5. XAI visualizations: Grad-CAM, LIME6. Complete training reports and comparisons

In [ ]:
# Install required packages (uncomment if needed)# !pip install torch torchvision opencv-python facenet-pytorch grad-cam lime scikit-learn tqdm matplotlib seaborn pandas pillow

In [ ]:
# Core importsimport osimport sysimport timeimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathfrom tqdm.auto import tqdmimport cv2from PIL import Imageimport jsonfrom collections import defaultdictimport gcfrom datetime import datetime# Deep Learningimport torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderimport torchvisionfrom torchvision import transforms, models# Metricsfrom sklearn.metrics import (    accuracy_score, f1_score, confusion_matrix,    classification_report, roc_auc_score, roc_curve, auc)from sklearn.preprocessing import label_binarizewarnings.filterwarnings('ignore')sns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (12, 8)# Seedsnp.random.seed(42)torch.manual_seed(42)if torch.cuda.is_available():    torch.cuda.manual_seed_all(42)# Devicedevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Using device: {device}')if torch.cuda.is_available():    print(f'GPU: {torch.cuda.get_device_name(0)}')    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Configurationclass Config:    # Paths    TRAIN_DIR = 'train'    TEST_DIR = 'test'    OUTPUT_DIR = 'outputs'    MODEL_DIR = 'models'    VIZ_DIR = 'visualizations'        # Classes    CLASSES = ['neutral', 'surprise', 'sad', 'happy', 'fear', 'disgust', 'angry']    NUM_CLASSES = 7        # Model params    IMG_SIZE = 224    BATCH_SIZE = 16    NUM_EPOCHS = 50    LEARNING_RATE = 0.001    EARLY_STOP_PATIENCE = 10        # Video params    MAX_FRAMES = 16    FRAME_SAMPLING = 'uniform'        # Face detection    FACE_MARGIN = 20    MIN_FACE_SIZE = 20        # Architectures to test    ARCHITECTURES = ['resnet18', 'resnet50', 'mobilenet_v2']config = Config()# Create directoriesfor dir_path in [config.OUTPUT_DIR, config.MODEL_DIR, config.VIZ_DIR]:    os.makedirs(dir_path, exist_ok=True)print(f'Configuration loaded')print(f'Classes: {config.CLASSES}')print(f'Architectures: {config.ARCHITECTURES}')

In [ ]:
# Face Detection Setuptry:    from facenet_pytorch import MTCNN    face_detector = MTCNN(        image_size=config.IMG_SIZE,        margin=config.FACE_MARGIN,        min_face_size=config.MIN_FACE_SIZE,        device=device,        post_process=False    )    print('MTCNN initialized')    USE_MTCNN = Trueexcept ImportError:    print('Using OpenCV Haar Cascade (install facenet-pytorch for MTCNN)')    face_cascade = cv2.CascadeClassifier(        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'    )    USE_MTCNN = Falsedef detect_and_crop_face(frame):    '''Detect and crop face from frame (RGB format)'''    if USE_MTCNN:        pil_img = Image.fromarray(frame)        face = face_detector(pil_img)        if face is not None:            face_np = face.permute(1, 2, 0).cpu().numpy()            face_np = ((face_np + 1) * 127.5).astype(np.uint8)            return face_np        return None    else:        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)        faces = face_cascade.detectMultiScale(            gray, 1.3, 5, minSize=(config.MIN_FACE_SIZE, config.MIN_FACE_SIZE)        )        if len(faces) > 0:            x, y, w, h = max(faces, key=lambda f: f[2] * f[3])            x1 = max(0, x - config.FACE_MARGIN)            y1 = max(0, y - config.FACE_MARGIN)            x2 = min(frame.shape[1], x + w + config.FACE_MARGIN)            y2 = min(frame.shape[0], y + h + config.FACE_MARGIN)            face = frame[y1:y2, x1:x2]            return cv2.resize(face, (config.IMG_SIZE, config.IMG_SIZE))        return Noneprint('Face detection ready')

In [ ]:
# Dynamic Image Generationdef generate_dynamic_image(frames, method='rank_pooling'):    '''Generate dynamic image from frames    Based on: https://github.com/tcvrick/dynamic-images-for-action-recognition    '''    if len(frames) == 0:        return None        frames = np.array(frames).astype(np.float32)    num_frames = len(frames)        if method == 'rank_pooling':        # Weighted sum with increasing weights        weights = np.arange(1, num_frames + 1) / num_frames        weights = weights / weights.sum()        dynamic_img = np.zeros_like(frames[0])        for i, frame in enumerate(frames):            dynamic_img += weights[i] * frame        return dynamic_img.astype(np.uint8)    else:        return np.mean(frames, axis=0).astype(np.uint8)print('Dynamic image generation ready')

In [ ]:
# Video Processingdef extract_frames(video_path, max_frames=None, sampling='uniform'):    '''Extract frames from video'''    cap = cv2.VideoCapture(str(video_path))    frames = []        while True:        ret, frame = cap.read()        if not ret:            break        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)        frames.append(frame_rgb)    cap.release()        if len(frames) == 0:        return []        if max_frames and len(frames) > max_frames:        if sampling == 'uniform':            indices = np.linspace(0, len(frames) - 1, max_frames, dtype=int)        else:  # random            indices = sorted(np.random.choice(len(frames), max_frames, replace=False))        frames = [frames[i] for i in indices]        return framesdef process_video_with_faces(video_path, max_frames=None):    '''Extract frames and detect faces'''    frames = extract_frames(video_path, max_frames, config.FRAME_SAMPLING)    face_frames = []        for frame in frames:        face = detect_and_crop_face(frame)        if face is not None:            face_frames.append(face)        # Fallback to resized frames if no faces detected    if len(face_frames) == 0 and len(frames) > 0:        face_frames = [cv2.resize(f, (config.IMG_SIZE, config.IMG_SIZE)) for f in frames]        return face_framesprint('Video processing ready')

In [ ]:
# Dataset Classclass EmotionDataset(Dataset):    def __init__(self, root_dir, transform=None, mode='static'):        '''        Args:            root_dir: Directory with class folders            transform: PyTorch transforms            mode: 'static', 'video', or 'dynamic'        '''        self.root_dir = Path(root_dir)        self.transform = transform        self.mode = mode        self.samples = []        self.labels = []                # Collect video files        for class_idx, class_name in enumerate(config.CLASSES):            class_dir = self.root_dir / class_name            if not class_dir.exists():                print(f'Warning: {class_dir} not found')                continue                        for video_file in class_dir.glob('*.mp4'):                self.samples.append(str(video_file))                self.labels.append(class_idx)                print(f'{mode.capitalize()} dataset: {len(self.samples)} videos')        def __len__(self):        return len(self.samples)        def __getitem__(self, idx):        video_path = self.samples[idx]        label = self.labels[idx]                if self.mode == 'static':            # Middle frame            frames = process_video_with_faces(video_path, max_frames=1)            img = frames[0] if frames else np.zeros((config.IMG_SIZE, config.IMG_SIZE, 3), dtype=np.uint8)            img = Image.fromarray(img)            if self.transform:                img = self.transform(img)            return img, label                elif self.mode == 'dynamic':            # Dynamic image            frames = process_video_with_faces(video_path, max_frames=config.MAX_FRAMES)            if frames:                dynamic_img = generate_dynamic_image(frames)            else:                dynamic_img = np.zeros((config.IMG_SIZE, config.IMG_SIZE, 3), dtype=np.uint8)            img = Image.fromarray(dynamic_img)            if self.transform:                img = self.transform(img)            return img, label                elif self.mode == 'video':            # Multiple frames            frames = process_video_with_faces(video_path, max_frames=config.MAX_FRAMES)            if not frames:                frames = [np.zeros((config.IMG_SIZE, config.IMG_SIZE, 3), dtype=np.uint8)]                        # Pad/trim to MAX_FRAMES            while len(frames) < config.MAX_FRAMES:                frames.append(frames[-1])            frames = frames[:config.MAX_FRAMES]                        # Convert to tensor            frames_tensor = []            for frame in frames:                img = Image.fromarray(frame)                if self.transform:                    img = self.transform(img)                frames_tensor.append(img)                        return torch.stack(frames_tensor), labelprint('Dataset class ready')

In [ ]:
# Data Transformstrain_transform = transforms.Compose([    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),    transforms.RandomHorizontalFlip(),    transforms.RandomRotation(10),    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])test_transform = transforms.Compose([    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])print('Transforms ready')

In [ ]:
# Model Architecturesdef get_image_model(arch='resnet18', pretrained=True):    '''Get CNN model for static/dynamic images'''    if arch == 'resnet18':        model = models.resnet18(pretrained=pretrained)        model.fc = nn.Linear(model.fc.in_features, config.NUM_CLASSES)    elif arch == 'resnet50':        model = models.resnet50(pretrained=pretrained)        model.fc = nn.Linear(model.fc.in_features, config.NUM_CLASSES)    elif arch == 'mobilenet_v2':        model = models.mobilenet_v2(pretrained=pretrained)        model.classifier[1] = nn.Linear(model.classifier[1].in_features, config.NUM_CLASSES)    elif arch == 'vgg16':        model = models.vgg16(pretrained=pretrained)        model.classifier[6] = nn.Linear(4096, config.NUM_CLASSES)    else:        raise ValueError(f'Unknown arch: {arch}')    return modelclass VideoLSTMModel(nn.Module):    '''Video model with CNN + LSTM'''    def __init__(self, arch='resnet18', hidden_size=512, num_layers=2):        super().__init__()                # Feature extractor        if arch == 'resnet18':            backbone = models.resnet18(pretrained=True)            self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])            feature_size = 512        elif arch == 'resnet50':            backbone = models.resnet50(pretrained=True)            self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])            feature_size = 2048        elif arch == 'mobilenet_v2':            backbone = models.mobilenet_v2(pretrained=True)            self.feature_extractor = backbone.features            self.pool = nn.AdaptiveAvgPool2d(1)            feature_size = 1280        else:            backbone = models.resnet18(pretrained=True)            self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])            feature_size = 512                self.arch = arch        self.lstm = nn.LSTM(feature_size, hidden_size, num_layers, batch_first=True)        self.fc = nn.Linear(hidden_size, config.NUM_CLASSES)        def forward(self, x):        # x: (batch, time, channels, height, width)        batch_size, timesteps, c, h, w = x.size()                # Extract features        x = x.view(batch_size * timesteps, c, h, w)        features = self.feature_extractor(x)                if self.arch == 'mobilenet_v2':            features = self.pool(features)                features = features.view(batch_size, timesteps, -1)                # LSTM        lstm_out, _ = self.lstm(features)                # Use last output        return self.fc(lstm_out[:, -1, :])print('Model architectures ready')

In [ ]:
# Training Functiondef train_epoch(model, loader, criterion, optimizer, device):    '''Train for one epoch'''    model.train()    running_loss = 0.0    all_preds = []    all_labels = []        for inputs, labels in tqdm(loader, desc='Training', leave=False):        inputs, labels = inputs.to(device), labels.to(device)                optimizer.zero_grad()        outputs = model(inputs)        loss = criterion(outputs, labels)        loss.backward()        optimizer.step()                running_loss += loss.item() * inputs.size(0)        _, preds = torch.max(outputs, 1)        all_preds.extend(preds.cpu().numpy())        all_labels.extend(labels.cpu().numpy())        epoch_loss = running_loss / len(loader.dataset)    epoch_acc = accuracy_score(all_labels, all_preds)        return epoch_loss, epoch_accdef validate(model, loader, criterion, device):    '''Validate model'''    model.eval()    running_loss = 0.0    all_preds = []    all_labels = []    all_probs = []        with torch.no_grad():        for inputs, labels in tqdm(loader, desc='Validating', leave=False):            inputs, labels = inputs.to(device), labels.to(device)                        outputs = model(inputs)            loss = criterion(outputs, labels)                        running_loss += loss.item() * inputs.size(0)            probs = F.softmax(outputs, dim=1)            _, preds = torch.max(outputs, 1)                        all_preds.extend(preds.cpu().numpy())            all_labels.extend(labels.cpu().numpy())            all_probs.extend(probs.cpu().numpy())        epoch_loss = running_loss / len(loader.dataset)    epoch_acc = accuracy_score(all_labels, all_preds)        return epoch_loss, epoch_acc, all_preds, all_labels, np.array(all_probs)print('Training functions ready')

In [ ]:
# Training Loopdef train_model(model, train_loader, val_loader, model_name, num_epochs=None):    '''Complete training loop with tracking'''    if num_epochs is None:        num_epochs = config.NUM_EPOCHS        model = model.to(device)    criterion = nn.CrossEntropyLoss()    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)    scheduler = optim.lr_scheduler.ReduceLROnPlateau(        optimizer, mode='min', factor=0.5, patience=5, verbose=True    )        history = {        'train_loss': [], 'train_acc': [],        'val_loss': [], 'val_acc': [],        'epoch_times': [], 'gpu_memory': []    }        best_val_acc = 0.0    best_model_wts = None    patience_counter = 0        print(f'\nTraining {model_name}')    print('=' * 60)        for epoch in range(num_epochs):        epoch_start = time.time()                # Train        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)                # Validate        val_loss, val_acc, _, _, _ = validate(model, val_loader, criterion, device)                # Scheduler step        scheduler.step(val_loss)                # Track GPU memory        gpu_mem = 0        if torch.cuda.is_available():            gpu_mem = torch.cuda.max_memory_allocated(device) / 1e9            torch.cuda.reset_peak_memory_stats(device)                epoch_time = time.time() - epoch_start                # Update history        history['train_loss'].append(train_loss)        history['train_acc'].append(train_acc)        history['val_loss'].append(val_loss)        history['val_acc'].append(val_acc)        history['epoch_times'].append(epoch_time)        history['gpu_memory'].append(gpu_mem)                print(f'Epoch {epoch+1}/{num_epochs}:')        print(f'  Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')        print(f'  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')        print(f'  Time: {epoch_time:.2f}s, GPU Mem: {gpu_mem:.2f}GB')                # Save best model        if val_acc > best_val_acc:            best_val_acc = val_acc            best_model_wts = model.state_dict().copy()            patience_counter = 0        else:            patience_counter += 1                # Early stopping        if patience_counter >= config.EARLY_STOP_PATIENCE:            print(f'Early stopping at epoch {epoch+1}')            break        # Load best weights    if best_model_wts is not None:        model.load_state_dict(best_model_wts)        # Save model    torch.save(model.state_dict(), f'{config.MODEL_DIR}/{model_name}.pth')        return model, historyprint('Training loop ready')

In [ ]:
# Comprehensive Evaluationdef evaluate_model(model, test_loader, model_name):    '''Comprehensive model evaluation'''    model.eval()    all_preds = []    all_labels = []    all_probs = []        with torch.no_grad():        for inputs, labels in tqdm(test_loader, desc='Testing'):            inputs = inputs.to(device)            outputs = model(inputs)            probs = F.softmax(outputs, dim=1)            _, preds = torch.max(outputs, 1)                        all_preds.extend(preds.cpu().numpy())            all_labels.extend(labels.numpy())            all_probs.extend(probs.cpu().numpy())        all_preds = np.array(all_preds)    all_labels = np.array(all_labels)    all_probs = np.array(all_probs)        # Metrics    acc = accuracy_score(all_labels, all_preds)    macro_f1 = f1_score(all_labels, all_preds, average='macro')        # Per-class metrics    report = classification_report(        all_labels, all_preds,         target_names=config.CLASSES,        output_dict=True    )        # Confusion matrix    cm = confusion_matrix(all_labels, all_preds)        # ROC-AUC per class    y_bin = label_binarize(all_labels, classes=range(config.NUM_CLASSES))    roc_auc_per_class = {}        for i, class_name in enumerate(config.CLASSES):        try:            roc_auc_per_class[class_name] = roc_auc_score(y_bin[:, i], all_probs[:, i])        except:            roc_auc_per_class[class_name] = 0.0        results = {        'model_name': model_name,        'accuracy': acc,        'macro_f1': macro_f1,        'classification_report': report,        'confusion_matrix': cm,        'roc_auc_per_class': roc_auc_per_class,        'predictions': all_preds,        'labels': all_labels,        'probabilities': all_probs    }        print(f'\n{model_name} Test Results:')    print(f'  Accuracy: {acc:.4f}')    print(f'  Macro-F1: {macro_f1:.4f}')        return resultsprint('Evaluation functions ready')

In [ ]:
# Visualization Functionsdef plot_training_curves(history, model_name):    '''Plot training/validation curves'''    fig, axes = plt.subplots(2, 2, figsize=(15, 10))        # Loss    axes[0, 0].plot(history['train_loss'], label='Train')    axes[0, 0].plot(history['val_loss'], label='Validation')    axes[0, 0].set_title(f'{model_name} - Loss')    axes[0, 0].set_xlabel('Epoch')    axes[0, 0].set_ylabel('Loss')    axes[0, 0].legend()    axes[0, 0].grid(True)        # Accuracy    axes[0, 1].plot(history['train_acc'], label='Train')    axes[0, 1].plot(history['val_acc'], label='Validation')    axes[0, 1].set_title(f'{model_name} - Accuracy')    axes[0, 1].set_xlabel('Epoch')    axes[0, 1].set_ylabel('Accuracy')    axes[0, 1].legend()    axes[0, 1].grid(True)        # Epoch time    axes[1, 0].plot(history['epoch_times'])    axes[1, 0].set_title(f'{model_name} - Training Time per Epoch')    axes[1, 0].set_xlabel('Epoch')    axes[1, 0].set_ylabel('Time (s)')    axes[1, 0].grid(True)        # GPU memory    axes[1, 1].plot(history['gpu_memory'])    axes[1, 1].set_title(f'{model_name} - GPU Memory Usage')    axes[1, 1].set_xlabel('Epoch')    axes[1, 1].set_ylabel('Memory (GB)')    axes[1, 1].grid(True)        plt.tight_layout()    plt.savefig(f'{config.VIZ_DIR}/{model_name}_training_curves.png', dpi=150, bbox_inches='tight')    plt.show()def plot_confusion_matrix(cm, model_name):    '''Plot confusion matrix'''    plt.figure(figsize=(10, 8))    sns.heatmap(        cm, annot=True, fmt='d', cmap='Blues',        xticklabels=config.CLASSES,        yticklabels=config.CLASSES    )    plt.title(f'{model_name} - Confusion Matrix')    plt.ylabel('True Label')    plt.xlabel('Predicted Label')    plt.tight_layout()    plt.savefig(f'{config.VIZ_DIR}/{model_name}_confusion_matrix.png', dpi=150, bbox_inches='tight')    plt.show()def plot_roc_curves(results, model_name):    '''Plot per-class ROC curves'''    y_bin = label_binarize(results['labels'], classes=range(config.NUM_CLASSES))    probs = results['probabilities']        plt.figure(figsize=(12, 10))        for i, class_name in enumerate(config.CLASSES):        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])        roc_auc = auc(fpr, tpr)        plt.plot(fpr, tpr, label=f'{class_name} (AUC = {roc_auc:.3f})')        plt.plot([0, 1], [0, 1], 'k--', label='Random')    plt.xlabel('False Positive Rate')    plt.ylabel('True Positive Rate')    plt.title(f'{model_name} - Per-Class ROC Curves')    plt.legend(loc='lower right')    plt.grid(True)    plt.tight_layout()    plt.savefig(f'{config.VIZ_DIR}/{model_name}_roc_curves.png', dpi=150, bbox_inches='tight')    plt.show()def visualize_dynamic_image_features(dataset, num_samples=5):    '''Visualize dynamic image generation'''    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))        for i in range(min(num_samples, len(dataset))):        video_path = dataset.samples[i]        label_idx = dataset.labels[i]        label = config.CLASSES[label_idx]                # Extract frames        frames = process_video_with_faces(video_path, max_frames=16)        if not frames:            continue                # First frame        axes[i, 0].imshow(frames[0])        axes[i, 0].set_title(f'{label} - First Frame')        axes[i, 0].axis('off')                # Middle frame        mid_idx = len(frames) // 2        axes[i, 1].imshow(frames[mid_idx])        axes[i, 1].set_title('Middle Frame')        axes[i, 1].axis('off')                # Last frame        axes[i, 2].imshow(frames[-1])        axes[i, 2].set_title('Last Frame')        axes[i, 2].axis('off')                # Dynamic image        dynamic_img = generate_dynamic_image(frames)        axes[i, 3].imshow(dynamic_img)        axes[i, 3].set_title('Dynamic Image')        axes[i, 3].axis('off')        plt.tight_layout()    plt.savefig(f'{config.VIZ_DIR}/dynamic_image_visualization.png', dpi=150, bbox_inches='tight')    plt.show()print('Visualization functions ready')

In [ ]:
# XAI - Grad-CAM Implementationclass GradCAM:    '''Grad-CAM for visualization'''    def __init__(self, model, target_layer):        self.model = model        self.target_layer = target_layer        self.gradients = None        self.activations = None                # Register hooks        target_layer.register_forward_hook(self.save_activation)        target_layer.register_backward_hook(self.save_gradient)        def save_activation(self, module, input, output):        self.activations = output.detach()        def save_gradient(self, module, grad_input, grad_output):        self.gradients = grad_output[0].detach()        def generate_cam(self, input_image, class_idx=None):        '''Generate Grad-CAM heatmap'''        self.model.eval()                # Forward pass        output = self.model(input_image)                if class_idx is None:            class_idx = output.argmax(dim=1).item()                # Backward pass        self.model.zero_grad()        one_hot = torch.zeros_like(output)        one_hot[0, class_idx] = 1        output.backward(gradient=one_hot, retain_graph=True)                # Generate CAM        weights = self.gradients.mean(dim=(2, 3), keepdim=True)        cam = (weights * self.activations).sum(dim=1, keepdim=True)        cam = F.relu(cam)        cam = F.interpolate(cam, size=input_image.shape[2:], mode='bilinear', align_corners=False)        cam = cam.squeeze().cpu().numpy()        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)                return cam, class_idxdef apply_gradcam_visualization(model, dataset, num_samples=5, arch='resnet18'):    '''Apply Grad-CAM to samples'''    # Get target layer    if 'resnet' in arch:        target_layer = model.layer4[-1]    elif 'mobilenet' in arch:        target_layer = model.features[-1]    else:        return        gradcam = GradCAM(model, target_layer)        fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))        for i in range(min(num_samples, len(dataset))):        img, label = dataset[i]        input_tensor = img.unsqueeze(0).to(device)                # Generate CAM        cam, pred_idx = gradcam.generate_cam(input_tensor)                # Denormalize image        img_np = img.cpu().numpy().transpose(1, 2, 0)        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])        img_np = np.clip(img_np, 0, 1)                # Plot original        axes[i, 0].imshow(img_np)        axes[i, 0].set_title(f'Original\nTrue: {config.CLASSES[label]}')        axes[i, 0].axis('off')                # Plot heatmap        axes[i, 1].imshow(cam, cmap='jet')        axes[i, 1].set_title(f'Grad-CAM\nPred: {config.CLASSES[pred_idx]}')        axes[i, 1].axis('off')                # Plot overlay        axes[i, 2].imshow(img_np)        axes[i, 2].imshow(cam, cmap='jet', alpha=0.5)        axes[i, 2].set_title('Overlay')        axes[i, 2].axis('off')        plt.tight_layout()    plt.savefig(f'{config.VIZ_DIR}/gradcam_visualization.png', dpi=150, bbox_inches='tight')    plt.show()print('Grad-CAM implementation ready')

In [ ]:
# Main Experiment Runner# Check if dataset existsif not os.path.exists(config.TRAIN_DIR) or not os.path.exists(config.TEST_DIR):    print('=' * 60)    print('DATASET NOT FOUND!')    print('=' * 60)    print('Please ensure you have the following directory structure:')    print('  train/')    print('    neutral/')    print('    surprise/')    print('    sad/')    print('    happy/')    print('    fear/')    print('    disgust/')    print('    angry/')    print('  test/')    print('    neutral/')    print('    surprise/')    print('    ...')    print('\nEach folder should contain MP4 video files.')    print('=' * 60)else:    print('Dataset directories found!')    print(f'Train: {config.TRAIN_DIR}')    print(f'Test: {config.TEST_DIR}')

In [ ]:
# Run All Experimentsall_results = {}all_histories = {}# Modes to testmodes = ['static', 'dynamic', 'video']for mode in modes:    print(f'\n{"="*70}')    print(f'RUNNING EXPERIMENTS FOR: {mode.upper()} MODE')    print(f'{"="*70}\n')        # Create datasets    if os.path.exists(config.TRAIN_DIR):        train_dataset = EmotionDataset(config.TRAIN_DIR, train_transform, mode=mode)        test_dataset = EmotionDataset(config.TEST_DIR, test_transform, mode=mode)                train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,                                  shuffle=True, num_workers=2)        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE,                                 shuffle=False, num_workers=2)                # Test each architecture        for arch in config.ARCHITECTURES:            model_name = f'{mode}_{arch}'            print(f'\n--- {model_name} ---')                        try:                # Create model                if mode in ['static', 'dynamic']:                    model = get_image_model(arch, pretrained=True)                else:  # video                    model = VideoLSTMModel(arch=arch)                                # Train                model, history = train_model(model, train_loader, test_loader, model_name)                all_histories[model_name] = history                                # Evaluate                results = evaluate_model(model, test_loader, model_name)                all_results[model_name] = results                                # Visualizations                plot_training_curves(history, model_name)                plot_confusion_matrix(results['confusion_matrix'], model_name)                plot_roc_curves(results, model_name)                                # Grad-CAM for image models                if mode in ['static', 'dynamic'] and 'resnet' in arch:                    apply_gradcam_visualization(model, test_dataset, num_samples=5, arch=arch)                                # Free memory                del model                torch.cuda.empty_cache()                gc.collect()                            except Exception as e:                print(f'Error with {model_name}: {str(e)}')                continue    else:        print(f'Skipping {mode} - dataset not found')print('\n' + '='*70)print('ALL EXPERIMENTS COMPLETED')print('='*70)

In [ ]:
# Visualize Dynamic Image Generation Processif os.path.exists(config.TRAIN_DIR):    print('Visualizing dynamic image generation...')    dynamic_dataset = EmotionDataset(config.TRAIN_DIR, None, mode='dynamic')    visualize_dynamic_image_features(dynamic_dataset, num_samples=5)else:    print('Dataset not found - skipping dynamic image visualization')

In [ ]:
# Final Comparison Tabledef create_comparison_table(results_dict):    '''Create comprehensive comparison table'''    comparison_data = []        for model_name, results in results_dict.items():        mode = model_name.split('_')[0]        arch = '_'.join(model_name.split('_')[1:])                row = {            'Mode': mode.capitalize(),            'Architecture': arch,            'Top-1 Accuracy': f"{results['accuracy']:.4f}",            'Macro-F1': f"{results['macro_f1']:.4f}",        }                # Add per-class AUC        for class_name, auc_score in results['roc_auc_per_class'].items():            row[f'{class_name}_AUC'] = f"{auc_score:.4f}"                comparison_data.append(row)        df = pd.DataFrame(comparison_data)        # Save to CSV    df.to_csv(f'{config.OUTPUT_DIR}/model_comparison.csv', index=False)        # Display    print('\n' + '='*100)    print('FINAL MODEL COMPARISON')    print('='*100)    print(df.to_string(index=False))    print('='*100)        # Create visual comparison    fig, axes = plt.subplots(1, 2, figsize=(16, 6))        # Accuracy comparison    df_plot = df.copy()    df_plot['Accuracy'] = df_plot['Top-1 Accuracy'].astype(float)    df_plot['Model'] = df_plot['Mode'] + '_' + df_plot['Architecture']        axes[0].barh(df_plot['Model'], df_plot['Accuracy'])    axes[0].set_xlabel('Accuracy')    axes[0].set_title('Model Accuracy Comparison')    axes[0].grid(True, axis='x')        # F1 comparison    df_plot['F1'] = df_plot['Macro-F1'].astype(float)    axes[1].barh(df_plot['Model'], df_plot['F1'])    axes[1].set_xlabel('Macro-F1')    axes[1].set_title('Model Macro-F1 Comparison')    axes[1].grid(True, axis='x')        plt.tight_layout()    plt.savefig(f'{config.VIZ_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')    plt.show()        return dfif all_results:    comparison_df = create_comparison_table(all_results)else:    print('No results available for comparison')

In [ ]:
# Print Detailed Classification Reportsprint('\n' + '='*100)print('DETAILED CLASSIFICATION REPORTS')print('='*100)for model_name, results in all_results.items():    print(f'\n{"-"*100}')    print(f'Model: {model_name}')    print(f'{"-"*100}')        report_df = pd.DataFrame(results['classification_report']).transpose()    print(report_df.to_string())        # Per-class ROC-AUC    print(f'\nPer-class ROC-AUC:')    for class_name, auc_score in results['roc_auc_per_class'].items():        print(f'  {class_name:12s}: {auc_score:.4f}')print('\n' + '='*100)

In [ ]:
# Training Statistics Summaryprint('\n' + '='*100)print('TRAINING STATISTICS')print('='*100)for model_name, history in all_histories.items():    print(f'\n{model_name}:')    print(f'  Total Epochs: {len(history["train_loss"])}')    print(f'  Avg Epoch Time: {np.mean(history["epoch_times"]):.2f}s')    print(f'  Total Training Time: {sum(history["epoch_times"])/60:.2f} min')    print(f'  Best Val Accuracy: {max(history["val_acc"]):.4f}')    print(f'  Final Train Loss: {history["train_loss"][-1]:.4f}')    print(f'  Final Val Loss: {history["val_loss"][-1]:.4f}')        if history['gpu_memory']:        print(f'  Peak GPU Memory: {max(history["gpu_memory"]):.2f} GB')print('='*100)

In [ ]:
# Error Analysis - Error Bucketsdef analyze_errors(results, model_name):    '''Analyze misclassifications'''    preds = results['predictions']    labels = results['labels']        # Find misclassifications    errors = preds != labels    error_indices = np.where(errors)[0]        print(f'\n{model_name} Error Analysis:')    print(f'Total errors: {len(error_indices)} / {len(labels)} ({100*len(error_indices)/len(labels):.2f}%)')        # Error buckets by true class    print('\nErrors by True Class:')    for i, class_name in enumerate(config.CLASSES):        class_mask = labels == i        class_errors = errors[class_mask]        n_errors = class_errors.sum()        n_total = class_mask.sum()                if n_total > 0:            error_rate = 100 * n_errors / n_total            print(f'  {class_name:12s}: {n_errors:3d} / {n_total:3d} ({error_rate:5.2f}%)')        # Confusion pairs    print('\nMost Common Misclassifications:')    error_pairs = []    for idx in error_indices:        true_label = config.CLASSES[labels[idx]]        pred_label = config.CLASSES[preds[idx]]        error_pairs.append((true_label, pred_label))        from collections import Counter    pair_counts = Counter(error_pairs)        for (true_label, pred_label), count in pair_counts.most_common(10):        print(f'  {true_label:12s} -> {pred_label:12s}: {count:3d}')print('\n' + '='*100)print('ERROR ANALYSIS')print('='*100)for model_name, results in all_results.items():    analyze_errors(results, model_name)print('='*100)

## Summary### Experiment Complete!All three approaches have been trained and evaluated:1. **Static Image Models** - Single frame extraction with face detection2. **Dynamic Image Models** - Temporal aggregation using rank pooling3. **Video Models** - LSTM-based temporal modeling### Outputs Generated:1. **Models**: Saved in `models/` directory2. **Visualizations**:    - Training curves (loss, accuracy, time, GPU memory)   - Confusion matrices   - Per-class ROC curves   - Grad-CAM heatmaps   - Dynamic image feature visualization3. **Reports**:   - Model comparison table   - Classification reports per class   - Error analysis and buckets   - Training statistics### Next Steps:1. Review the comparison table to identify best-performing approach2. Analyze confusion matrices to understand error patterns3. Examine Grad-CAM visualizations for model interpretation4. Consider ensemble methods combining multiple approaches5. Fine-tune hyperparameters for best model### Key Metrics Reported:- ✅ Top-1 Accuracy on test set- ✅ Macro-F1 score (handles class imbalance)- ✅ Per-class ROC-AUC curves- ✅ Confusion matrices- ✅ Training/validation curves- ✅ Training time per epoch- ✅ GPU memory usage- ✅ Classification reports per class- ✅ XAI visualizations (Grad-CAM)- ✅ Error buckets and analysis- ✅ Dynamic image feature visualization

In [ ]:
# Save all results to JSON# Prepare results for JSON (convert numpy arrays to lists)results_for_json = {}for model_name, results in all_results.items():    results_for_json[model_name] = {        'accuracy': float(results['accuracy']),        'macro_f1': float(results['macro_f1']),        'roc_auc_per_class': {k: float(v) for k, v in results['roc_auc_per_class'].items()},        'classification_report': results['classification_report']    }# Save resultswith open(f'{config.OUTPUT_DIR}/all_results.json', 'w') as f:    json.dump(results_for_json, f, indent=2)# Save historieshistories_for_json = {}for model_name, history in all_histories.items():    histories_for_json[model_name] = {        'train_loss': [float(x) for x in history['train_loss']],        'train_acc': [float(x) for x in history['train_acc']],        'val_loss': [float(x) for x in history['val_loss']],        'val_acc': [float(x) for x in history['val_acc']],        'epoch_times': [float(x) for x in history['epoch_times']],        'gpu_memory': [float(x) for x in history['gpu_memory']]    }with open(f'{config.OUTPUT_DIR}/training_histories.json', 'w') as f:    json.dump(histories_for_json, f, indent=2)print('\nAll results saved to:')print(f'  - {config.OUTPUT_DIR}/all_results.json')print(f'  - {config.OUTPUT_DIR}/training_histories.json')print(f'  - {config.OUTPUT_DIR}/model_comparison.csv')print(f'\nVisualizations saved to: {config.VIZ_DIR}/')print(f'Models saved to: {config.MODEL_DIR}/')